In [ ]:
!pip install ultralytics
!pip install roboflow

tập dữ liệu 1k

In [ ]:
!pip install roboflow

import os
from roboflow import Roboflow
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("fire-detection-lfqdr").project("wildfire_segmentation")
version = project.version(2)
dataset = version.download("yolov8")


In [ ]:
from ultralytics import YOLO
model_seg = YOLO('yolov8n-seg.pt') # Load model Nano Segment

In [ ]:
model_seg.train(
    data=f"{dataset.location}/data.yaml",
    epochs=300,
    imgsz=640,
    patience=30,
    batch=64,
)

In [ ]:
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
import glob
import random
import numpy as np
from google.colab.patches import cv2_imshow # Thư viện hiển thị ảnh của Colab
model_path = '/content/runs/segment/train4/weights/best.pt'
test_images_path = '/content/ARDIN-2-1/test/images/*.jpg'
try:
    model = YOLO(model_path)
    print(f"Đã load model: {model_path}")
except:
    print("Không tìm thấy model! Kiểm tra lại đường dẫn model_path.")
    # Fallback về model gốc nếu chưa train xong để demo code
    model = YOLO('yolov8n-seg.pt')
img_list = glob.glob(test_images_path)
if len(img_list) > 0:
    img_path = random.choice(img_list)
    print(f"Đang test ảnh: {img_path}")
    results = model.predict(img_path, conf=0.25)
    result = results[0]
    img_plotted = result.plot()
    total_area_pixels = 0
    if result.masks is not None:
        # Lấy danh sách các mặt nạ (masks)
        masks = result.masks.data.cpu().numpy()
        for mask in masks:
            # Đếm số pixel cháy (giá trị > 0)
            area = np.count_nonzero(mask)
            total_area_pixels += area

        print(f"Phát hiện đám cháy!")
        print(f"Tổng diện tích cháy (Pixels): {total_area_pixels}")
        # Giả sử 1 pixel = 1 mét vuông (cần chỉnh theo GSD thực tế)
        print(f"Ước tính diện tích thực: {total_area_pixels} m² (Giả định GSD=1)")
    else:
        print("Không phát hiện đám cháy trong ảnh này.")

    # 5. Hiển thị ảnh
    cv2_imshow(img_plotted)

else:
    print("Không tìm thấy ảnh nào trong folder test!")